In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / 'src'))

import stim
import _cultiv_noise as cultiv_noise

circuit = stim.Circuit.from_file(
    str(Path.cwd().parent / 'in/circuits/honeycomb_gidney/model=SDEM3,r=16,w=10,h=15,p=0.001,obs=H,sheared=False.stim')
)
print(circuit.diagram('matchgraph-3d'))

In [ ]:
raw = circuit.without_noise()
print('=== BEFORE noise ===')
print(raw)

model = cultiv_noise.NoiseModel.flexible_pairwise(p_m_bulk=0.001, p_spam=0.001, p_depolarizing=0.001)
noisy = model.noisy_circuit(raw)
print('\n=== AFTER noise ===')
print(noisy)

In [ ]:
import sinter
import numpy as np
import matplotlib.pyplot as plt

# Compare the original (SDEM3, p baked into the circuit) gidney honeycomb
# against the same circuit re-noised with the *balanced* flexible_pairwise
# model (p_m_bulk = p_spam = p_depolarizing = p).
p_values = [0.001]

raw = circuit.without_noise()

tasks = [
    sinter.Task(
        circuit=circuit,
        json_metadata={'noise_model': 'SDEM3', 'p': 0.001},
    ),
]
for p in p_values:
    model = cultiv_noise.NoiseModel.flexible_pairwise(
        p_m_bulk=p, p_spam=p, p_depolarizing=p,
    )
    tasks.append(sinter.Task(
        circuit=model.noisy_circuit(raw),
        json_metadata={'noise_model': 'flexible_pairwise_balanced', 'p': float(p)},
    ))

results = sinter.collect(
    num_workers=8,
    max_shots=200_000,
    max_errors=200,
    tasks=tasks,
    decoders=['pymatching'],
    print_progress=True,
)

fig, ax = plt.subplots(1, 1, figsize=(6, 5))
sinter.plot_error_rate(
    ax=ax,
    stats=results,
    x_func=lambda s: s.json_metadata['p'],
    group_func=lambda s: s.json_metadata['noise_model'],
    failure_units_per_shot_func=lambda s: 1,
)
ax.loglog()
ax.set_xlabel('p')
ax.set_ylabel('Logical error rate per shot')
ax.grid(which='both', alpha=0.4)
ax.legend(loc='lower right')
fig.tight_layout()
plt.show()